<p style="text-align: left;">
  <img
    src="https://raw.githubusercontent.com/poma-ai/.github/main/assets/POMA_AI_Logo_Pink.svg"
    alt="POMA AI Logo"
    height="40"
    style="vertical-align: middle;"
  />
    <span style="display: inline-block;    margin: 0 18px;    font-size: 42px;    font-weight: 800;    line-height: 1;    transform: scale(1.5) translateY(-2px);    vertical-align: middle;">×</span>
  <img
    src="https://qdrant.tech/img/qdrant-logo.svg"
    alt="Qdrant Logo"
    height="40"
    style="vertical-align: middle;"
  />
</p>




---

## Overview

> - **Qdrant** is *the leading* open source *vector database and similarity search engine* designed to handle high-dimensional vectors for performance and massive AI applications, advanced filtering, and production‑grade scalability.
> - **POMA AI** specializes in *document chunking that preserves semantic and structural context*, producing semantically coherent chunksets which retain layout, hierarchy, and contextual boundaries, making them well‑suited for ingestion into RAG workflows and scores with the Cheatsheet algorithm for deduplication and document content sorting after retrieval.
>
>Together, **POMA** prepares documents in a retrieval‑optimal form, while **Qdrant** stores and
>indexes those chunks efficiently for downstream search and RAG pipelines.

This document describes how to connect **POMA's** chunking output to a new or existing **Qdrant** setup
and ingest the resulting chunksets into a collection with automatic embeddings.

---

## Getting Started

### Get yourself a POMA API Key

Create or copy a POMA API key from the POMA Playground.

- 1. Open https://app.poma-ai.com/

- 2. Navigate to the top right to register or login.


        >  <figure style="margin: 8px 0 0 0;">
        >    <img
        >      src="../../assets/qdrant/Poma_01.png"
        >      alt="POMA Register Screenshot"
        >      style="max-width: 680px; width: 100%; height: auto; border-radius: 8px; display: block;"
        >    />
        >  </figure>


- 3. Enter your E-Mail and insert the verification code sent to it.

- 4. After login you'll find "API Keys" in the navigation bar on the left; klick on it, copy your API key and set it as Environment variable `POMA_API_KEY`.


        >  <figure style="margin: 8px 0 0 0;">
        >    <img
        >      src="../../assets/qdrant/Poma_02.png"
        >      alt="POMA API Key Screenshot"
        >      style="max-width: 680px; width: 100%; height: auto; border-radius: 8px; display: block;"
        >    />
        >  </figure>



---

## Installation

We made the installation simple by just adding theis package:

In [ ]:
%pip install -qq "poma[qdrant]"
%pip install -qq python-dotenv

Loading neccesary imports:

In [ ]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

from poma import Poma
from poma.integrations.qdrant import PomaQdrant, QdrantConfig, InferenceConfig, VectorConfig
# -------------------- Load example files if in colab --------------------
import sys; base="https://raw.githubusercontent.com/poma-ai/.github/main/notebooks/qdrant"; ("google.colab" in sys.modules) and [__import__("urllib.request").request.urlretrieve(f"{base}/{f}", f) for f in ("example.pdf","example.poma")]

---
## Necessary Credentials and API keys

### Set your POMA API key and Qdrant Credentials

For Qdrant Cloud, use a cluster endpoint and cluster API key from your dashboard.

https://cloud.qdrant.io/

Additional setup details can be found in the Qdrant managed cloud docs:
https://qdrant.tech/documentation/cloud/ 

In [ ]:
# Set Qdrant credentials
def secret(name: str) -> str:
    clean = lambda s: s.strip().strip('"\'')
    try:
        from google.colab import userdata  # type: ignore
        v = os.getenv(name) or userdata.get(name)
    except Exception:
        v = os.getenv(name)
    v = clean(v) if v else clean(getpass.getpass(f"{name}: "))
    if not v:
        raise ValueError(f"{name} is required but empty")
    os.environ[name] = v
    return v

POMA_API_KEY = secret("POMA_API_KEY")
QDRANT_URL = secret("QDRANT_URL")
QDRANT_API_KEY = secret("QDRANT_API_KEY")

print("✅ Credentials loaded")

#### Optional Credentials

In [ ]:
OPENAI_API_KEY = secret("OPENAI_API_KEY")

In [ ]:
OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY")

---

## Ingesting files and process with POMA AI 

#### Setup POMA client 

In [ ]:
client = Poma(os.environ["POMA_API_KEY"])

#### Process file with the poma pileline

In [ ]:
job = client.start_chunk_file("example.pdf")
chunk_result = client.get_chunk_result(job["job_id"], show_progress=True, download_dir="./", filename="example.poma")
print(chunk_result)

#### Or select ready *.poma (example) file instead

In [ ]:
chunk_result = "example.poma"

---

## Ingesting POMA AI files into Qdrant

Fully-managed cloud inference, with a lot of configuration knobs

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_hybrid_explicit_1"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="inference_hybrid",
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,
        client_kwargs={"timeout": 120},
    ),
    vectors=VectorConfig(dense_name="dense", sparse_name="sparse"),
    inference_config=InferenceConfig(
        dense_model="openrouter/thenlper/gte-base",
        dense_options={"dimensions": 768},
        sparse_model="Qdrant/bm25",
    ),
    fusion_k=60,
    fusion_candidates=100,
    fusion_weights=(0.8, 0.2),  # prioritize dense a bit more
)

### Ingest results in the Qdrant collection

In [ ]:
poma_qdrant.ingest(chunk_result)

## Get structure preserving cheatsheets

In [ ]:
cheatsheets = poma_qdrant.search_cheatsheets("Who are the authors of this paper?", limit=3)

for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Cheatsheet {i} ===")
    print(f"file_id: {cs['file_id']}")
    print(f"tag: {cs['tag']}")
    print("content:")
    print(cs["content"])

---

## Appendix



#### More constructor examples (increasing specificity)

The examples below go from minimal defaults to fully explicit advanced configuration.



##### Example 1: Memory inference minimal (auto default mode)

In [ ]:
QDRANT_COLLECTION_NAME="in_memory_fast_embed"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
)

##### Example 2: Persisted path inference minimal (auto default mode)

In [ ]:
QDRANT_COLLECTION_NAME="local_fastembed_path_persistent"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="fastembed",  # explicit (default is also fastembed when cloud_inference=False)
    qdrant=QdrantConfig(
        mode="path",
        path="./.qdrant_data",  # persistent local storage folder
    ),
)

##### Example 3:  Setup the PomaQdrant client for the RAG (basic example for online inference)

In [ ]:
QDRANT_COLLECTION_NAME="online_fast_embed"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    qdrant=QdrantConfig(
        url=os.environ["QDRANT_URL"],
        api_key=os.environ.get("QDRANT_API_KEY"),
    )
)

#### Setup the PomaQdrant client for the RAG (basic example for cloud inference, defaults to OpenAI embeddings)

OPENAI_API_KEY=**required!**

##### Example 4: Cloud inference minimal (auto default mode)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_auto_default"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,  # auto-selects inference_dense if embedding_mode is omitted
    ),
)

##### Example 5: Explicit cloud dense mode + model (dimension only in vectors)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_dense_vectors_only"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="inference_dense",
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,
    ),
    vectors=VectorConfig(dense_size=1536),  # single source of truth for dimension
    inference_config=InferenceConfig(
        dense_model="openai/text-embedding-3-large",
        # openai-api-key comes from OPENAI_API_KEY env if not passed here
    ),
)

##### Example 6: Explicit cloud dense mode + dimensions only in InferenceConfig

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_dense_inference_only"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="inference_dense",
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,
    ),
    inference_config=InferenceConfig(
        dense_model="openai/text-embedding-3-large",
        dense_options={
            "dimensions": 1536,  # SDK mirrors this into vectors.dense_size
            # "openai-api-key": os.environ["OPENAI_API_KEY"],  # optional if env var is set
        },
    ),
)

##### Example 7: Cloud sparse inference - experiment, not reccomended for general usage (no embedding API key needed)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_sparse_only"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="inference_sparse",
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,
    ),
    inference_config=InferenceConfig(
        sparse_model="Qdrant/bm25",
    ),
)

##### Example 8: Cloud hybrid inference (dense + sparse + fusion tuning)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_hybrid_explicit"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="inference_hybrid",
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL_CLOUD"],
        api_key=os.environ.get("QDRANT_API_KEY_CLOUD"),
        cloud_inference=True,
    ),
    vectors=VectorConfig(dense_size=1536, dense_name="dense", sparse_name="sparse"),
    inference_config=InferenceConfig(
        dense_model="openai/text-embedding-3-large",
        dense_options={"dimensions": 1536},
        sparse_model="Qdrant/bm25",
    ),
    fusion_k=60,
    fusion_candidates=100,
    fusion_weights=(0.7, 0.3),  # prioritize dense a bit more
)

##### Example 9: Non-cloud fastembed (self-hosted/local endpoint)

You can also host an qdrant instance locally, to do so change the `QDRANT_URL` to the local url.
This example would except in colab, if you replace the LOCAL envs.

In [ ]:
QDRANT_COLLECTION_NAME = "local_fastembed"
LOCAL_QDRANT_URL = "http://localhost:6333"
LOCAL_QDRANT_API_KEY = "1234567890"

poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    qdrant=QdrantConfig(
        mode="url",
        url=os.environ["QDRANT_URL"], # replace with LOCAL_QDRANT_URL
        api_key=os.environ.get("QDRANT_API_KEY"), # replace with LOCAL_QDRANT_API_KEY
        cloud_inference=False,  # keeps fastembed mode valid
    ),
    embedding_mode="fastembed",
    inference_config=InferenceConfig(
        dense_model="jinaai/jina-embeddings-v2-base-en",
        sparse_model="Qdrant/bm25",
    ),
)

##### Example 10: Persisted storage with external embeddings

Additional dependencies needed in this example

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI # type: ignore
from typing import Sequence

def openai_dense_embed(texts: Sequence[str]) -> list[list[float]]:
    res = OpenAI().embeddings.create(
        model="text-embedding-3-small",  # 1536 dims
        input=list(texts),
    )
    return [item.embedding for item in res.data]

QDRANT_COLLECTION_NAME="external_openai_dense"
poma_qdrant = PomaQdrant(
    collection_name=QDRANT_COLLECTION_NAME,
    embedding_mode="external_dense",
    qdrant=QdrantConfig(
        mode="path",
        path="./.qdrant_data",
        cloud_inference=False,  # not using Qdrant cloud inference
    ),
    dense_embed_fn=openai_dense_embed,
    vectors=VectorConfig(dense_size=1536, dense_name="dense"),
)

##### Run each example with the below ingestion and cheatsheet search.

In [ ]:
poma_qdrant.ingest(chunk_result)

cheatsheets = poma_qdrant.search_cheatsheets("Who are the authors of this paper?", limit=3)
for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Cheatsheet {i} ===")
    print(f"file_id: {cs['file_id']}")
    print(f"tag: {cs['tag']}")
    print("content:")
    print(cs["content"])

##### Prints FastEmbed models

In [ ]:
from qdrant_client import QdrantClient
l = list(QdrantClient.list_text_models().keys()); print("FastEmbed dense models:", *[l[i:i+3] for i in range(0, len(l), 3)], sep="\n")
l = list(QdrantClient.list_sparse_models().keys()); print("FastEmbed sparse models:", *[l[i:i+3] for i in range(0, len(l), 3)], sep="\n")

---

## Notes

- Self‑hosted Qdrant deployments may not require an API key.
- Embeddings are generated automatically using Qdrant's built‑in embedding support.

##### Config knobs explained

- `embedding_mode`: Selects retrieval strategy (`fastembed`, `external_*`, `inference_*`).
- `cloud_inference`: Enables Qdrant Cloud Inference behavior and defaults.
- `inference_config.dense_model` / `sparse_model`: Chooses dense and sparse inference models.
- `dense_options["dimensions"]`: Provider-side dense embedding output size (for OpenRouter, OpenAI, etc.).
- `vectors.dense_size`: Collection vector size stored in Qdrant schema.
- Dimension rule: set it in one place, or in both places with the same value; mismatches raise a clear exception.